# B2.9 · Building a domain harness: one skeleton, four oracles

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *AI for Security*

Builds on **[B2.8 · Idempotency, replay and rollback](https://spbreed.github.io/cyber-commons/lessons/B2.8.html)**.

| | |
|---|---|
| Open-source tooling | OpenGrep, OWASP ZAP, OWASP Threat Dragon, CAI |
| Open-weight models | GLM-4.6, Kimi K2 |
| Frontier models | Claude Sonnet 5 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Four teams build four harnesses and each re-decides loop control, budgets and verification from scratch. The loops end up nearly identical. What actually differs is the oracle and the blast radius — which belong to the domain, not to the loop.

> **At CyberTravels.** CyberTravels needs SAST, DAST, a threat model and a pentest against TripBot. They are one skeleton with four different oracles, and building four harnesses is how four teams each get the oracle wrong differently.

## 2 · The framework

```
   one skeleton                       four oracles
   +---------------------+            sast   reachable + failing test
   | plan . act . verify |  <-------  tmodel present in new, absent in old
   | budget . stop       |            dast   response differs from control
   +---------------------+            ptest  a shell, a row, a file
             |
      blast radius, per domain
      read-only -> replica-write -> live action (needs a signed scope)
```

Teams build a SAST harness, then a threat-modelling harness, then a DAST
harness, then a pentest harness — and re-decide loop control, budgets, retries
and verification four times. The loops end up nearly identical, and the four
teams each get one thing wrong in their own way.

They are the same skeleton. Plan a candidate, act on it, verify it, stop. What
actually differs between the four is two things, and both belong to the domain
rather than to the loop:

**The oracle** — what decides a candidate is real. Reachability plus a failing
test for static analysis. A diff against the previous model for threat
modelling. An observed change in a response for DAST. A shell, a row or a file
for a pentest. The oracle is the whole value of the harness: everything else is
plumbing you have already built once.

**The blast radius** — what acting costs if the candidate is wrong. Reading
source costs nothing. Sending a request to a replica costs a little. Running an
exploit against a live host costs an incident, so it needs an authorisation the
loop cannot grant itself.

Build the skeleton once. Then a new domain is an oracle, a blast radius, and
nothing else.

## 3 · One skeleton, and the two things a domain supplies

In [ ]:
DOMAINS = {
 "sast":         {"reads": "source at a commit",     "blast": "read-only"},
 "threat model": {"reads": "architecture and IaC",   "blast": "read-only"},
 "dast":         {"reads": "a running replica",      "blast": "replica-write"},
 "pentest":      {"reads": "an owned host in scope", "blast": "live-action"},
}
ORACLES = {
 "sast":         ("reachable from an entrypoint AND a failing test",
                  lambda e: e["reachable"] and e["failing_test"]),
 "threat model": ("present in the new model and absent from the old",
                  lambda e: e["in_new"] and not e["in_old"]),
 "dast":         ("response differs from the control request",
                  lambda e: e["response_differs"]),
 "pentest":      ("an artefact that should not have been obtainable",
                  lambda e: e["artefact"] is not None),
}
AUTHORISED = {"read-only", "replica-write"}      # live-action needs a signed scope

def harness(domain, candidates, oracle, budget=6, scope_signed=False):
    """The skeleton. Identical for all four domains."""
    blast = DOMAINS[domain]["blast"]
    if blast not in AUTHORISED and not scope_signed:
        return {"domain": domain, "refused": "live action without a signed scope",
                "confirmed": [], "steps": 0}
    confirmed, steps = [], 0
    for c in sorted(candidates, key=lambda c: c["id"]):
        if steps >= budget:
            break
        steps += 1
        if oracle(c["evidence"]):
            confirmed.append(c)
    return {"domain": domain, "refused": None, "confirmed": confirmed, "steps": steps}

for d in sorted(DOMAINS):
    print(f"{d:14s}{DOMAINS[d]['blast']:14s}oracle: {ORACLES[d][0]}")

## 4 · Four domains through the same loop

In [ ]:
CANDIDATES = {
 "sast": [
  {"id": "unit_07 CWE-89 in build_query", "real": True,
   "evidence": {"reachable": True,  "failing_test": True}},
  {"id": "unit_12 CWE-89 in log_line",    "real": False,
   "evidence": {"reachable": False, "failing_test": False}},
  {"id": "unit_31 CWE-22 in export_path", "real": True,
   "evidence": {"reachable": True,  "failing_test": True}}],
 "threat model": [
  {"id": "worker -> db crosses trust 0 to 2", "real": True,
   "evidence": {"in_new": True,  "in_old": False}},
  {"id": "component web renamed to frontend", "real": False,
   "evidence": {"in_new": True,  "in_old": True}},
  {"id": "/admin/export on an untrusted entry", "real": True,
   "evidence": {"in_new": True,  "in_old": False}}],
 "dast": [
  {"id": "GET /v1/users returns 200 unauthenticated", "real": True,
   "evidence": {"response_differs": True}},
  {"id": "banner discloses framework version",        "real": False,
   "evidence": {"response_differs": False}},
  {"id": "POST /report reflects payload unencoded",   "real": True,
   "evidence": {"response_differs": True}}],
 "pentest": [
  {"id": "password auth permits spraying, no lockout", "real": True,
   "evidence": {"artefact": "session as svc-reports"}},
  {"id": "expired certificate on the partner CDN",     "real": False,
   "evidence": {"artefact": None}},
  {"id": "/backup listing exposes a database dump",    "real": True,
   "evidence": {"artefact": "orders.sql, 41 MB"}}],
}

def precision(result, candidates):
    if not result["confirmed"]:
        return None
    return sum(c["real"] for c in result["confirmed"]) / len(result["confirmed"])

print(f"{'domain':14s}{'confirmed':>10}{'precision':>11}  note")
for d in sorted(DOMAINS):
    r = harness(d, CANDIDATES[d], ORACLES[d][1], scope_signed=True)
    p = precision(r, CANDIDATES[d])
    print(f"{d:14s}{len(r['confirmed']):>10}{p:>11.2f}  {ORACLES[d][0][:38]}")

r = harness("pentest", CANDIDATES["pentest"], ORACLES["pentest"][1])
print(f"\npentest without a signed scope: {r['refused']}")
assert r["confirmed"] == []

## 5 · Where it breaks — the oracle everyone reaches for

Every one of those four oracles is a fact about the target. The tempting fifth is the model's own opinion, because it is the only one that works in every domain without being built.

In [ ]:
def model_oracle(evidence):
    """The model read the evidence and is confident. Confidence is not an oracle."""
    return True

print(f"{'domain':14s}{'confirmed':>10}{'precision':>11}")
for d in sorted(DOMAINS):
    r = harness(d, CANDIDATES[d], model_oracle, scope_signed=True)
    print(f"{d:14s}{len(r['confirmed']):>10}{precision(r, CANDIDATES[d]):>11.2f}")

total = sum(len(CANDIDATES[d]) for d in CANDIDATES)
real = sum(c["real"] for d in CANDIDATES for c in CANDIDATES[d])
print(f"\nevery one of {total} candidates confirmed; {real} of them are real.")
print("Precision 0.67 in every domain, and it looks like 1.00 from inside the")
print("harness, because the thing producing the finding is also the thing")
print("agreeing with it.")
assert all(precision(harness(d, CANDIDATES[d], model_oracle, scope_signed=True),
                     CANDIDATES[d]) < 1.0 for d in DOMAINS)

## 6 · The control — classify the oracle, gate on the class

In [ ]:
ORACLE_CLASS = {
 "sast":         "deterministic",   # re-runs to the same answer on the same commit
 "threat model": "deterministic",
 "dast":         "observational",   # a real observation, but of a system that moves
 "pentest":      "observational",
 "model":        "judgement",       # not re-checkable, not falsifiable
}
MAY_FILE = {"deterministic", "observational"}

def dispatch(domain, oracle_name, result):
    cls = ORACLE_CLASS[oracle_name]
    return {"domain": domain, "oracle_class": cls,
            "action": "file the finding" if cls in MAY_FILE else "queue for a human",
            "count": len(result["confirmed"])}

for d in sorted(DOMAINS):
    good = harness(d, CANDIDATES[d], ORACLES[d][1], scope_signed=True)
    bad  = harness(d, CANDIDATES[d], model_oracle, scope_signed=True)
    print(dispatch(d, d, good))
    print(dispatch(d, "model", bad))

print()
print("The skeleton did not change once across four domains. What changed was")
print("the oracle and the blast radius - which is the whole argument for")
print("building the loop once and never again.")
assert dispatch("sast", "model", harness("sast", CANDIDATES["sast"], model_oracle,
                scope_signed=True))["action"] == "queue for a human"

## What you just proved

One skeleton runs all four domains unchanged. With each domain's own oracle every confirmed finding is real — precision 1.00 across sast, threat model, dast and pentest — and the pentest run refuses outright without a signed scope, because its blast radius is live action. Swapping in the model's own confidence as the oracle confirms all 12 candidates, 8 of which are real: precision 0.67 in every domain, invisible from inside the harness.

## Your turn

Name the oracle for the harness you are building. If the sentence contains the words 'the model determines', you have a judgement, not an oracle — and the finding belongs in a human queue rather than in a ticket.

---

**Next → [B2.10 · Choosing the model backbone](https://spbreed.github.io/cyber-commons/lessons/B2.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*